<a href="https://colab.research.google.com/github/ddahlgren11/MoneyMaker/blob/main/actualcapstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Add the dependencies to use tweepy for twiiter and alpaca for my stock data

In [ ]:
!pip install tweepy alpaca-py pandas

Code for pulling data from twitter

In [ ]:
import tweepy
import pandas as pd
from google.colab import userdata
import time

# 1. Setup Authentication
BEARER_TOKEN = userdata.get('TWITTER_BEARER_TOKEN')
client = tweepy.Client(bearer_token=BEARER_TOKEN, wait_on_rate_limit=True)

# 2. Define your list of CEOs and their corresponding companies
# (Matching your US-1: Configure CEOs and companies requirement)
ceo_targets = {
    "elonmusk": "Tesla",
    "tim_cook": "Apple",
    "satyanadella": "Microsoft"
}

all_tweets = []

# 3. Loop through each CEO
for username, company in ceo_targets.items():
    try:
        # Resolve username to User ID
        user = client.get_user(username=username)
        user_id = user.data.id
        print(f"Fetching tweets for {username} (ID: {user_id})...")

        # Fetch Tweets with specific fields for analysis
        # (Supports US-7: Sentiment and US-10: Impact Score)
        response = client.get_users_tweets(
            id=user_id,
            max_results=5,
            tweet_fields=['created_at', 'public_metrics', 'lang']
        )

        if response.data:
            for tweet in response.data:
                all_tweets.append({
                    'ceo': username,
                    'company': company,
                    'tweet_id': tweet.id,
                    'text': tweet.text,
                    'created_at': tweet.created_at,
                    'retweets': tweet.public_metrics['retweet_count'],
                    'likes': tweet.public_metrics['like_count'],
                    'replies': tweet.public_metrics['reply_count']
                })

        # Optional: brief sleep to be safe with rate limits
        time.sleep(1)

    except Exception as e:
        print(f"Error fetching {username}: {e}")

# 4. Final Dataset
df_all_ceos = pd.DataFrame(all_tweets)
print(f"\nCollection Complete! Total tweets gathered: {len(df_all_ceos)}")
display(df_all_ceos.head())

# 5. Save to Google Drive immediately
# (Supports US-6: Clean and validate data)
df_all_ceos.to_csv('/content/drive/MyDrive/raw_ceo_tweets_multi.csv', index=False)

Fetching tweets for elonmusk (ID: 44196397)...


Pulling ALPACA date

In [ ]:
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from google.colab import userdata
from datetime import datetime

# 1. Authenticate using Colab Secrets
API_KEY = userdata.get('ALPACA_API_KEY')
SECRET_KEY = userdata.get('ALPACA_SECRET_KEY')

# Initialize the data client
data_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)

# 2. Fetch Historical Stock Data (e.g., Tesla for Elon Musk)
# We use TimeFrame.Day to align with your "next-day prediction" goal
request_params = StockBarsRequest(
    symbol_or_symbols="TSLA",
    timeframe=TimeFrame.Day,
    start=datetime(2025, 1, 1)  # Set your start date
)

# 3. Retrieve and Convert to a DataFrame
bars = data_client.get_stock_bars(request_params)
df_stocks = bars.df
display(df_stocks.head())

### Aggregate Daily Tweet Sentiment

To align with the daily stock data, we need to aggregate the sentiment scores of tweets for each CEO on a daily basis. We'll calculate the daily average sentiment score for each CEO.

In [ ]:
import pandas as pd

# Convert 'created_at' to datetime and extract the date
df_all_ceos['date'] = pd.to_datetime(df_all_ceos['created_at']).dt.date

# Calculate daily average sentiment for each CEO
df_daily_sentiment = df_all_ceos.groupby(['ceo', 'date'])['sentiment_score'].mean().reset_index()
df_daily_sentiment = df_daily_sentiment.rename(columns={'sentiment_score': 'avg_daily_sentiment'})

print("Daily average sentiment per CEO calculated:")
display(df_daily_sentiment.head())